In [3]:
import torch
import torch.nn as nn

# Loss 1 — MSELoss (regression)

In [4]:
mse = nn.MSELoss()

# Model predicted these values
predictions = torch.tensor([2.5, 3.0, 4.5])

# Actual true values
targets     = torch.tensor([3.0, 3.0, 5.0])

In [5]:
loss = mse(predictions, targets)
print(loss)   # tensor(0.1667)

# Manual check: mean of squared differences
# ((2.5-3)² + (3-3)² + (4.5-5)²) / 3
# = (0.25 + 0 + 0.25) / 3 = 0.1667  ✅

tensor(0.1667)


# Loss 2 — BCEWithLogitsLoss (binary classification)

In [6]:
bce = nn.BCEWithLogitsLoss()

# Raw model outputs (logits) — not probabilities yet
# Positive = model leans toward class 1, negative = leans toward class 0
logits  = torch.tensor([ 2.1, -1.3,  0.8, -0.5])

# True labels: 1 = positive, 0 = negative
targets = torch.tensor([ 1.0,  0.0,  1.0,  0.0])

In [7]:
loss = bce(logits, targets)
print(loss)   # tensor(0.3085) — a fairly confident, mostly correct model

tensor(0.3004)


# Loss 3 — CrossEntropyLoss (multi-class)

In [12]:
ce = nn.CrossEntropyLoss()

# Model outputs: raw scores for each class (3 samples, 4 classes)
# Each row = one sample's scores across all classes
logits = torch.tensor([
    [2.0, 1.0, 0.1, 0.5],   # sample 1: model strongly leans toward class 0
    [0.5, 2.5, 0.3, 0.1],   # sample 2: model strongly leans toward class 1
    [0.1, 0.2, 3.1, 0.4],   # sample 3: model strongly leans toward class 2
])

# True class indices (not one-hot — just the class number)
targets = torch.tensor([0, 1, 2])   # all three predictions were correct

In [9]:
loss = ce(logits, targets)
print(loss)   # tensor(0.3625) — low loss, model was right and confident

tensor(0.3344)


In [10]:
# ⚠️ targets are class INDICES, not one-hot vectors
targets = torch.tensor([0, 1, 2])    # ✅ correct
targets = torch.tensor([[1,0,0],     # ❌ wrong — CrossEntropyLoss doesn't want this
                         [0,1,0],
                         [0,0,1]])

In [13]:
# ⚠️ logits go IN — not probabilities
# Don't do this:
probs = torch.softmax(logits, dim=1)
loss  = ce(probs, targets)           # ❌ double-applying softmax internally

# Do this:
loss = ce(logits, targets)           # ✅ CrossEntropyLoss applies softmax itself


In [14]:
loss

tensor(0.3344)

In [15]:
ce = nn.CrossEntropyLoss()
target = torch.tensor([0])   # true class is 0

# Case 1: confident and CORRECT
logits = torch.tensor([[5.0, 0.1, 0.1]])
print(ce(logits, target))    # tensor(0.0134) — very low ✅

# Case 2: uncertain
logits = torch.tensor([[0.4, 0.3, 0.3]])
print(ce(logits, target))    # tensor(1.0116) — medium

# Case 3: confident and WRONG
logits = torch.tensor([[-4.0, 5.0, 0.1]])
print(ce(logits, target))    # tensor(9.0136) — very high ❌

tensor(0.0148)
tensor(1.0331)
tensor(9.0075)


The loss is harshest when you're confidently wrong. That's by design — it creates a strong gradient signal that forces large weight updates. Being uncertain gets penalised less because the model "knows it doesn't know."

## Putting it together — model + loss in one go

In [16]:
import torch
import torch.nn as nn

# A tiny classifier: 4 features → 3 classes
model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 3)    # 3 output neurons = 3 classes (no softmax — CE handles it)
)

In [21]:
# Note: nn.Sequential is a shortcut when you don't need custom forward() logic
# It just runs layers in order — same result as writing a full nn.Module class

loss_fn = nn.CrossEntropyLoss()

# A batch of 5 samples, 4 features each
x      = torch.rand(5, 4)
target = torch.tensor([0, 2, 1, 0, 2])   # true class for each sample

In [22]:
# Forward pass
logits = model(x)               # shape: (5, 3)

# Compute loss
loss = loss_fn(logits, target)
print(loss)   # a single scalar — how wrong the model is right now

tensor(1.1339, grad_fn=<NllLossBackward0>)
